# MassSpecGym → ICEBERG trên Kaggle

Notebook này dựng tập `msg_simulation` của MassSpecGym từ đầu và (tuỳ chọn) train thử ICEBERG-Gen:

1. Cài môi trường (torch 2.6 + CUDA 12.4, DGL, PyG, RDKit…)
2. Clone repo `ms-pred`
3. Tải `MassSpecGym1.5.tsv` từ Hugging Face
4. Tạo `labels.tsv` + `splits/split.tsv` (simulation challenge, có năng lượng va chạm)
5. Chuyển phổ thành `spec_files.hdf5`
6. Chạy MAGMA + subformula (tạo nhãn cho ICEBERG)
7. (Tuỳ chọn) Train thử ICEBERG-Gen

**Trước khi chạy:** Settings → *Internet: On*, *Accelerator: GPU T4* (GPU chỉ cần cho bước 7).
Chạy lần lượt từ trên xuống. **Đừng `import torch` trước ô cài đặt.**

## 0. Cấu hình

In [ ]:
import os

REPO_URL = "https://github.com/coleygroup/ms-pred.git"  # đổi sang fork của bạn nếu cần
REPO_DIR = "/kaggle/working/ms-pred"                     # ô 5 ghi converter vào đường dẫn này
DATASET  = "msg_simulation"
TMP_DIR  = "/tmp/msg_raw"                                # file TSV gốc (không lưu vào output)

# Số phổ dùng. MAGMA rất tốn CPU: hãy chạy thử với số nhỏ để đo tốc độ,
# rồi mới đặt None (= toàn bộ ~119k phổ, có thể mất nhiều giờ với 4 CPU của Kaggle).
N_SPECS  = 2000
WORKERS  = os.cpu_count()

RUN_TRAIN  = False   # True để train thử ICEBERG-Gen ở bước 7
MAX_EPOCHS = 2
print("CPU:", WORKERS)

## 1. Cài môi trường

Mất khoảng 5–10 phút. Không dùng `download.pytorch.org` (Kaggle hay không phân giải được tên miền này):
torch 2.6.0 trên PyPI bản Linux đã kèm CUDA 12.4. DGL và torch-scatter lấy wheel dựng sẵn khớp torch 2.6 + cu124,
và **cấm build từ mã nguồn** để lỗi hiện ra ngay thay vì build hỏng.

Cảnh báo kiểu `torchtune ... requires torchdata==0.11.0` hay `torchaudio ... requires torch==...` là của các gói
Kaggle cài sẵn mà pipeline không dùng, bỏ qua được.

In [ ]:
import socket, subprocess, sys, time

# Kiểm tra mạng trước: nếu dòng nào báo LỖI, bật Settings → Internet: On rồi chạy lại ô này
for host in ["pypi.org", "data.dgl.ai", "data.pyg.org", "github.com", "huggingface.co"]:
    try:
        socket.gethostbyname(host)
        print("OK  ", host)
    except OSError:
        print("LỖI ", host, "(không phân giải được tên miền)")

def pip(*args, retries=3):
    for attempt in range(1, retries + 1):
        if subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args]).returncode == 0:
            return
        print(f"pip thất bại (lần {attempt}/{retries}), thử lại...")
        time.sleep(10)
    raise RuntimeError("Cài đặt thất bại: pip install " + " ".join(args))

pip("torch==2.6.0", "torchvision==0.21.0")
pip("dgl", "--only-binary", ":all:", "-f", "https://data.dgl.ai/wheels/torch-2.6/cu124/repo.html")
pip("torch-scatter", "--only-binary", ":all:", "-f", "https://data.pyg.org/whl/torch-2.6.0+cu124.html")
pip("rdkit==2025.03.*", "pytorch-lightning", "torchdata<0.10", "h5py", "pathos", "omegaconf", "einops",
    "linsatnet", "platformdirs", "seaborn", "scikit-learn", "pyyaml", "huggingface_hub")
pip("git+https://github.com/Thinklab-SJTU/pygmtools.git")

# Kiểm tra trong một tiến trình Python mới, tránh dính bản torch cũ đã nạp
subprocess.run([sys.executable, "-c",
    "import torch, dgl, torch_scatter, rdkit; "
    "print('torch', torch.__version__, '| CUDA build', torch.version.cuda, '| GPU', torch.cuda.is_available()); "
    "print('dgl', dgl.__version__, '| torch_scatter', torch_scatter.__version__, '| rdkit', rdkit.__version__)"],
    check=True)

## 2. Clone repo

In [ ]:
import sys, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
os.environ["PYTHONPATH"] = os.path.join(REPO_DIR, "src") + ":" + os.environ.get("PYTHONPATH", "")
print(os.getcwd())

In [ ]:
# Kiểm tra môi trường
import torch, dgl, torch_scatter, rdkit
import ms_pred.common as common
assert torch.__version__.startswith("2.6"), f"torch {torch.__version__}: hãy Restart kernel rồi chạy lại từ ô 0"
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| dgl", dgl.__version__, "| rdkit", rdkit.__version__)

## 3. Tải MassSpecGym1.5.tsv

In [ ]:
from huggingface_hub import hf_hub_download
TSV_PATH = hf_hub_download(
    repo_id="roman-bushuiev/MassSpecGym",
    filename="data/MassSpecGym1.5.tsv",
    repo_type="dataset",
    local_dir=TMP_DIR,
)
print(TSV_PATH, round(os.path.getsize(TSV_PATH) / 1e6), "MB")

## 4. Tạo labels.tsv và splits/split.tsv

Cùng quy tắc với `data_scripts/create_msg_simulation_dataset.py`: giữ dòng `simulation_challenge == True`
và có năng lượng va chạm. Script gốc đòi thêm bảng ứng viên retrieval (tải từ Dropbox), nên ở đây
tạo trực tiếp phần cần cho train.

In [ ]:
import pandas as pd
from pathlib import Path

data_dir = Path("data/spec_datasets") / DATASET
(data_dir / "splits").mkdir(parents=True, exist_ok=True)

cols = ["identifier", "smiles", "inchikey", "formula", "precursor_mz", "adduct",
        "instrument_type", "collision_energy", "fold", "simulation_challenge"]
df = pd.read_csv(TSV_PATH, sep="\t", usecols=cols)
print("Tổng số dòng:", len(df))

df = df[df["simulation_challenge"].astype(str).str.strip().str.lower().isin({"true", "1", "yes"})]
df["ce"] = pd.to_numeric(df["collision_energy"], errors="coerce")
df = df.dropna(subset=["ce"]).drop_duplicates(subset=["identifier"])
fold = df["fold"].astype(str).str.strip().str.lower()
df["split"] = fold.map(lambda f: "train" if f == "train" else ("val" if f in {"val", "valid", "validation"} else "test"))
print("Simulation challenge + có năng lượng:", len(df))   # kỳ vọng 119,029

if N_SPECS is not None and N_SPECS < len(df):
    df = df.groupby("split", group_keys=False).sample(frac=N_SPECS / len(df), random_state=0)
print("Dùng:", len(df), df["split"].value_counts().to_dict())

fmt_ce = lambda v: f"['{int(v)}']" if float(v).is_integer() else f"['{v:g}']"
labels = pd.DataFrame({
    "dataset": "MassSpecGym",
    "spec": df["identifier"].astype(str),
    "ionization": df["adduct"],
    "formula": df["formula"],
    "smiles": df["smiles"],
    "inchikey": df["inchikey"],
    "instrument": df["instrument_type"],
    "collision_energies": df["ce"].map(fmt_ce),
    "precursor": df["precursor_mz"],
})
labels.to_csv(data_dir / "labels.tsv", sep="\t", index=False)
pd.DataFrame({"name": labels["spec"], "split": df["split"].values}).to_csv(
    data_dir / "splits" / "split.tsv", sep="\t", index=False)
labels.head()

## 5. Chuyển phổ thành spec_files.hdf5
Ô dưới ghi script `create_msg_spec_files.py` vào repo (repo gốc chưa có script này), rồi chạy nó.

In [ ]:
%%writefile /kaggle/working/ms-pred/data_scripts/create_msg_spec_files.py
#!/usr/bin/env python3
"""Convert MassSpecGym1.5.tsv into the repository's spec_files.hdf5 layout.

Each spectrum is stored as one SIRIUS-style ``.ms`` string under the key
``{identifier}.ms``, which is what ``common.HDF5Dataset.read_str`` and
``common.parse_spectra`` expect (see run_magma.py and
data_scripts/forms/01_assign_subformulae.py). Example entry:

    >compound MassSpecGymID0000001
    >formula C16H17NO4
    >parentmass 288.1225
    >ionization [M+H]+
    >instrumentation Orbitrap
    >smiles CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC
    >InChIKey VFMQMACUYWGDOJ

    >collision 30
    91.0542 0.2316
    ...

Rows without a known collision energy are written under ``>collision nan``
(the key ICEBERG uses for missing energies) unless ``--missing-ce skip`` is set.
"""

from __future__ import annotations

import argparse
import json
import os
from pathlib import Path

import h5py
import pandas as pd


DEFAULT_SOURCE_TABLE = (
    "https://huggingface.co/datasets/roman-bushuiev/MassSpecGym/"
    "resolve/main/data/MassSpecGym1.5.tsv"
)
DEFAULT_OUTPUT = Path("data/spec_datasets/msg/spec_files.hdf5")

META_COLUMNS = {
    "formula": "formula",
    "parentmass": "precursor_mz",
    "ionization": "adduct",
    "instrumentation": "instrument_type",
    "smiles": "smiles",
    "InChIKey": "inchikey",
}


def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--source-table", default=DEFAULT_SOURCE_TABLE,
                        help="Path or URL of MassSpecGym1.5.tsv")
    parser.add_argument("--output", type=Path, default=DEFAULT_OUTPUT)
    parser.add_argument("--labels", type=Path, default=None,
                        help="Optional labels.tsv; only specs in its 'spec' column are written")
    parser.add_argument("--missing-ce", choices=["nan", "skip"], default="nan",
                        help="How to handle rows without a collision energy")
    parser.add_argument("--id-col", default="identifier")
    parser.add_argument("--mz-col", default="mzs")
    parser.add_argument("--inten-col", default="intensities")
    parser.add_argument("--ce-col", default="collision_energy")
    parser.add_argument("--chunksize", type=int, default=20000)
    parser.add_argument("--limit", type=int, default=None,
                        help="Stop after writing this many spectra (for quick checks)")
    parser.add_argument("--overwrite", action="store_true")
    parser.add_argument("--audit-out", type=Path, default=None)
    return parser.parse_args()


def format_collision_energy(value) -> str | None:
    """Return the energy label used in '>collision {label}', or None if unknown."""
    try:
        val = float(value)
    except (TypeError, ValueError):
        return None
    if pd.isna(val):
        return None
    return str(int(val)) if val.is_integer() else f"{val:g}"


def split_peaks(value) -> list[str]:
    if not isinstance(value, str):
        return []
    return [tok.strip() for tok in value.split(",") if tok.strip()]


def build_ms_string(row: pd.Series, args, ce_label: str) -> str | None:
    """Build one SIRIUS .ms string, or None if the peak lists are unusable."""
    mzs = split_peaks(row[args.mz_col])
    intens = split_peaks(row[args.inten_col])
    if len(mzs) == 0 or len(mzs) != len(intens):
        return None
    try:
        peaks = [(float(m), float(i)) for m, i in zip(mzs, intens)]
    except ValueError:
        return None

    header = [f">compound {row[args.id_col]}"]
    for key, col in META_COLUMNS.items():
        val = row.get(col)
        if val is not None and not pd.isna(val) and str(val).strip():
            header.append(f">{key} {str(val).strip()}")

    # The blank line separates metadata from the spectrum block; parse_spectra
    # relies on it to tell the two groups apart.
    peak_lines = [f"{m} {i}" for m, i in peaks]
    return "\n".join(header) + f"\n\n>collision {ce_label}\n" + "\n".join(peak_lines) + "\n"


def write_str(h5: h5py.File, name: str, data: str):
    """Same on-disk layout as common.HDF5Dataset.write_str."""
    dt = h5py.special_dtype(vlen=str)
    ds = h5.create_dataset(name, (1,), dtype=dt, compression="gzip")
    ds[0] = data


def main():
    args = get_args()

    if args.output.exists() and not args.overwrite:
        raise FileExistsError(f"{args.output} exists; pass --overwrite to replace it")

    keep_specs = None
    if args.labels is not None:
        keep_specs = set(pd.read_csv(args.labels, sep="\t", usecols=["spec"])["spec"].astype(str))

    required = {args.id_col, args.mz_col, args.inten_col, args.ce_col}
    audit = {
        "source_table": str(args.source_table),
        "source_rows": 0,
        "written": 0,
        "written_missing_ce": 0,
        "skipped_not_in_labels": 0,
        "skipped_missing_ce": 0,
        "skipped_bad_peaks": 0,
        "skipped_duplicate_id": 0,
    }
    seen = set()

    args.output.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = args.output.with_name(args.output.name + ".tmp")
    if tmp_path.exists():
        tmp_path.unlink()

    done = False
    with h5py.File(tmp_path, "w") as h5:
        for chunk in pd.read_csv(args.source_table, sep="\t", chunksize=args.chunksize):
            missing = required - set(chunk.columns)
            if missing:
                raise KeyError(f"Missing columns {sorted(missing)}; available: {list(chunk.columns)}")

            for _, row in chunk.iterrows():
                audit["source_rows"] += 1
                spec = str(row[args.id_col])

                if keep_specs is not None and spec not in keep_specs:
                    audit["skipped_not_in_labels"] += 1
                    continue
                if spec in seen:
                    audit["skipped_duplicate_id"] += 1
                    continue

                ce_label = format_collision_energy(row[args.ce_col])
                if ce_label is None:
                    if args.missing_ce == "skip":
                        audit["skipped_missing_ce"] += 1
                        continue
                    ce_label = "nan"

                ms_str = build_ms_string(row, args, ce_label)
                if ms_str is None:
                    audit["skipped_bad_peaks"] += 1
                    continue

                write_str(h5, f"{spec}.ms", ms_str)
                seen.add(spec)
                audit["written"] += 1
                if ce_label == "nan":
                    audit["written_missing_ce"] += 1

                if args.limit is not None and audit["written"] >= args.limit:
                    done = True
                    break
            if done:
                break

    os.replace(tmp_path, args.output)

    if keep_specs is not None:
        audit["labels_specs"] = len(keep_specs)
        audit["labels_specs_missing_from_output"] = len(keep_specs - seen)

    print(json.dumps(audit, indent=2))
    if args.audit_out is not None:
        args.audit_out.parent.mkdir(parents=True, exist_ok=True)
        args.audit_out.write_text(json.dumps(audit, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
!python data_scripts/create_msg_spec_files.py \
    --source-table {TSV_PATH} \
    --labels {data_dir}/labels.tsv \
    --output {data_dir}/spec_files.hdf5 \
    --audit-out {data_dir}/spec_files_audit.json \
    --overwrite

In [ ]:
# Đọc thử một phổ bằng parser của repo
h5 = common.HDF5Dataset(data_dir / "spec_files.hdf5")
name = next(iter(h5.get_all_names()))
meta, specs = common.parse_spectra(h5.read_str(name).split("\n"))
print(name, meta)
for ce, ms in specs.items():
    print("CE", ce, "| số peak:", ms.num_peaks)
h5.close()

## 6. MAGMA + subformula
Tạo `magma_outputs/magma_tree.hdf5` (nhãn cho ICEBERG-Gen) và `subformulae/no_subform.hdf5`. Đây là bước lâu nhất.

In [ ]:
import time
t0 = time.time()
!dataset={DATASET} workers={WORKERS} bash data_scripts/dag/run_magma.sh
print(f"Mất {(time.time() - t0) / 60:.1f} phút cho {len(labels)} phổ")

In [ ]:
for p in ["spec_files.hdf5", "magma_outputs/magma_tree.hdf5", "subformulae/no_subform.hdf5"]:
    f = data_dir / p
    print(f"{p:35s}", "OK" if f.exists() else "THIẾU", f"{f.stat().st_size / 1e6:.1f} MB" if f.exists() else "")

## 7. (Tuỳ chọn) Train thử ICEBERG-Gen

Siêu tham số lấy từ `configs/iceberg/msg_simulation/dag_train_msg_simulation.yaml`, chỉ giảm số epoch
và số worker cho Kaggle. Train đầy đủ (200 epoch, toàn bộ dữ liệu) vượt quá giới hạn 12 giờ của một phiên Kaggle.

In [ ]:
if RUN_TRAIN:
    gpu_flag = "--gpu" if torch.cuda.is_available() else ""
    !python src/ms_pred/iceberg/train_gen.py {gpu_flag} \
        --seed 1 --num-workers 2 --batch-size 32 --max-epochs {MAX_EPOCHS} \
        --dataset-name {DATASET} --split-name split.tsv --dataset-labels labels.tsv \
        --magma-folder magma_outputs --save-dir results/kaggle_{DATASET}/gen \
        --learning-rate 0.000996 --lr-decay-rate 0.7214 --dropout 0.2 \
        --mpnn-type GGNN --pe-embed-k 14 --pool-op avg --set-layers 0 \
        --hidden-size 512 --weight-decay 0.0 --layers 6 --root-encode gnn \
        --encode-forms --embed-adduct --embed-collision --embed-elem-group \
        --embed-instrument --add-hs
    !find results/kaggle_{DATASET} -name "*.ckpt"
else:
    print("Bỏ qua train (RUN_TRAIN = False)")

## Lưu kết quả

Mọi thứ nằm trong `/kaggle/working/ms-pred`. Bấm **Save Version** (Save & Run All) để Kaggle giữ lại thư mục
`data/spec_datasets/msg_simulation` và `results/`. Có thể biến output đó thành Kaggle Dataset để dùng lại
ở notebook khác, khỏi phải chạy MAGMA lần nữa.